# Fine-tune Llama 3.1 8B in Google Colab



In [1]:
!pip install -q accelerate==0.34.2 peft==0.12.0 bitsandbytes==0.43.3 transformers==4.43.1 trl==0.10.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [2]:
import sys

IN_COLAB = 'google.colab' in sys.modules
RUN_TRAINING_CELLS = IN_COLAB

EXPERIMENT_NAME = 'Head-QA-Gen-LLaMa3_1-8b/'
DRIVE_FOLDER_LOCATION = '/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/' + EXPERIMENT_NAME

In [3]:
# Mounting google drive
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Using my own Google Drive during the experiment to save all checkpoints and training logs.

if IN_COLAB:
    # Adapted from:  https://robertbrucecarter.com/writing/2020/06/setting-your-working-directory-to-google-drive-in-a-colab-notebook/
    import os

    def create_and_set_working_directory(path: str):
        # check if your project folder exists. if not, it will be created.
        if os.path.isdir(path) == False:
            os.mkdir(path)
            print(path + ' did not exist but was created.')

        # change the OS to use your project folder as the working directory
        os.chdir(path)

        print('Working directory changed to: \n' + path)

    create_and_set_working_directory(DRIVE_FOLDER_LOCATION)
    !pwd

Working directory changed to: 
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/Head-QA-Gen-LLaMa3_1-8b/
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/Head-QA-Gen-LLaMa3_1-8b


In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd
import numpy as np

In [6]:
from google.colab import userdata
sec_key = userdata.get('HF_TOKEN')

In [7]:
!huggingface-cli login --token $sec_key

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [8]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Meta-Llama-3.1-8B-Instruct"

# The instruction dataset to use
dataset_name = "head_qa"

# Fine-tuned model name
new_model = "llama-3-8b-spanish-qa-gen"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 2

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = True

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 100

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = 300

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

In [9]:
# Load dataset (you can process it here)
dataset_train = load_dataset(dataset_name, split="train", trust_remote_code=True)
dataset_test = load_dataset(dataset_name, split="test")
dataset_validation = load_dataset(dataset_name, split="validation")
dataset_train[2]

head_qa.py:   0%|          | 0.00/6.21k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

head-qa-es-en-pdfs.zip:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2657 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2742 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1366 [00:00<?, ? examples/s]

{'name': 'Cuaderno_2013_1_B',
 'year': '2013',
 'category': 'biology',
 'qid': 3,
 'qtext': 'NO generan potenciales de acción:',
 'ra': 2,
 'image': None,
 'answers': [{'aid': 1, 'atext': 'Fibras musculares lisas.'},
  {'aid': 2, 'atext': 'Neuronas bipolares de la retina.'},
  {'aid': 3, 'atext': 'Fibras musculares estriadas esqueléticas.'},
  {'aid': 4, 'atext': 'Fibras musculares cardíacas.'},
  {'aid': 5, 'atext': 'Neuronas ganglionares de la retina.'}]}

In [10]:
def flatten_dataset(dataset):
    data_df = pd.DataFrame(dataset)
    answers = pd.json_normalize(data_df['answers'])
    df_answers = pd.DataFrame()
    data_df['answer_text'] = None
    for c in range(len(answers.columns)):
        answer = pd.json_normalize(answers[c])
        df_answers = pd.concat([df_answers, answer['atext']], axis=1)
        df_answers.rename(columns={'atext': c}, inplace=True)
    for i in range(data_df.shape[0]):
        col = data_df.loc[i, 'ra']
        data_df.loc[i, 'answer_text'] = df_answers.iloc[i, col-1]
    return data_df

In [11]:
data_train_df = flatten_dataset(dataset_train)
data_test_df = flatten_dataset(dataset_test)
data_val_df = flatten_dataset(dataset_validation)
print(data_train_df.shape)
print(data_test_df.shape)
print(data_val_df.shape)
data_train_df.head()

(2657, 9)
(2742, 9)
(1366, 9)


,name,year,category,qid,qtext,ra,image,answers,answer_text
0,Cuaderno_2013_1_B,2013,biology,1,Los potenciales postsinápticos excitadores:,3,None,"[{'aid': 1, 'atext': 'Son de tipo todo o nada....",Se pueden sumar.
1,Cuaderno_2013_1_B,2013,biology,2,Placa motora es la unión entre la neurona moto...,2,None,"[{'aid': 1, 'atext': 'Músculo liso.'}, {'aid':...",Músculo esquelético.
2,Cuaderno_2013_1_B,2013,biology,3,NO generan potenciales de acción:,2,None,"[{'aid': 1, 'atext': 'Fibras musculares lisas....",Neuronas bipolares de la retina.
3,Cuaderno_2013_1_B,2013,biology,4,En la iniciación de los movimientos voluntario...,1,None,"[{'aid': 1, 'atext': 'Corteza premotora.'}, {'...",Corteza premotora.
4,Cuaderno_2013_1_B,2013,biology,5,Los corpúsculos de Pacini:,4,None,"[{'aid': 1, 'atext': 'Están inervados por fibr...",Se localizan en zonas profundas de la dermis.


In [12]:
SEP_TOKEN = ''
MASKING_CHANCE = 0.3 #30% chance to replace the answer with '[MASK]'

In [13]:
class QGDataset(Dataset):

    def __init__(
        self,
        data: pd.DataFrame,
        tokenizer: AutoTokenizer,
        source_max_token_len: int,
        target_max_token_len: int
        ):

        self.tokenizer = tokenizer
        self.data = data
        self.source_max_token_len = source_max_token_len
        self.target_max_token_len = target_max_token_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        data_row = self.data.iloc[index]

        if np.random.rand() > MASKING_CHANCE:
            answer = data_row['answer_text']
        else:
            answer = '[MASK]'

        source_encoding = tokenizer(
            '{} {} {}'.format(answer, SEP_TOKEN, data_row['qtext']),
            max_length= self.source_max_token_len,
            padding='max_length',
            truncation= True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        target_encoding = tokenizer(
            '{} {} {}'.format(data_row['answer_text'], SEP_TOKEN, data_row['qtext']),
            max_length=self.target_max_token_len,
            padding='max_length',
            truncation = True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        labels = target_encoding['input_ids']
        labels[labels == 0] = -100

        return dict(
            answer_text = data_row['answer_text'],
            # context = data_row['context'],
            question = data_row['qtext'],
            input_ids = source_encoding['input_ids'].flatten(),
            attention_mask = source_encoding['attention_mask'].flatten(),
            labels=labels.flatten()
            )

In [ ]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    return_dict=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training
train_dataset = QGDataset(
    data=data_train_df,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

Your GPU supports bfloat16: accelerate training with bf16=True


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Step,Training Loss
25,2.972800
50,2.420500
75,2.149200
100,2.159200
125,2.205500
150,2.109100
175,2.090000
200,2.089100
225,2.004600
250,2.011400


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir results/runs

In [ ]:
# Empty VRAM
del model
del pipe
del trainer
import gc
gc.collect()
gc.collect()

20933

In [14]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [15]:
test_dataset = QGDataset(
    data=data_test_df,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)
test_dataset[2]

{'answer_text': 'Locomoción celular.',
 'question': 'NO es una función de los filamentos intermedios:',
 'input_ids': tensor([128000,   9330,    316,   2168,   3244,  88226,     13,    220,   5782,
           1560,   5203,  64177,    409,   2537,   1488,  37879,    958,   2106,
           3614,     25, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 

In [22]:
from random import randint
rand_idx = randint(0, len(test_dataset))

'En un quimiostato inoculado con un cultivo puro, al aumentar el flujo disminuye constantemente:'

In [28]:
# Test on sample
prompt = test_dataset[rand_idx]["question"]
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("Original question:\n" + test_dataset[rand_idx]["question"])
print("Original answer:\n" + test_dataset[rand_idx]["answer_text"])
print(result[0]['generated_text'])

Original question:
En un quimiostato inoculado con un cultivo puro, al aumentar el flujo disminuye constantemente:
Original answer:
El tiempo de generación.
<s>[INST] En un quimiostato inoculado con un cultivo puro, al aumentar el flujo disminuye constantemente: [/INST]  El pH. La concentración de oxígeno. La velocidad de crecimiento. La producción de un metabolito secundario. La actividad enzimática.  La respuesta correcta es la 4.  En un quimiostato, que es un reactor con control de pH y temperatura, el pH, la temperatura y la concentración de oxígeno son controlados. ¿Qué respuesta es correcta? En un quimiostato inoculado con un cultivo puro, al aumentar el flujo disminuye constantemente: El pH. La concentración de oxígeno. La velocidad de crecimiento. La producción de un metabolito secundario. La actividad enzimática.  La respuesta correcta es la 4.  En un quimi


In [32]:
prompt = "¿Qué es un quimiostato inoculado?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] ¿Qué es un quimiostato inoculado? [/INST]  En el proceso de fermentación industrial de la levadura, el control de la concentración de oxígeno en el medio es fundamental para la producción de alcohol y otros productos de valor. Para lograrlo, se añaden a la fermentación pequeñas cantidades de un compuesto que se llama quimiostato. ¿Qué es un quimiostato inoculado? (1 punto) : ( )  Es un compuesto que se utiliza como inhibidor de crecimiento bacteriano. ( )  Es un compuesto que se utiliza para controlar la concentración de oxígeno en el medio de fermentación. ( )  Es un compuesto que se utiliza para controlar la concentración de CO2 en el medio de fermentación. ( )  Es un compuesto que se utiliza para controlar la concentración de N2 en el medio de


In [35]:
prompt = "Genera una pregunta sobre biolgía con tres opciones falsas y una verdadera."
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=180)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] Genera una pregunta sobre biolgía con tres opciones falsas y una verdadera. [/INST]  ¿Qué tipo de pregunta es la siguiente? En la fisiología del hombre, ¿cuál es la función del sistema nervioso simpático? A) Regula la temperatura corporal B) Regula la respiración C) Regula la frecuencia cardíaca D) Regula el estado de alerta de la persona E) Regula la respuesta al estrés. La respuesta correcta es D. ¿Qué tipo de pregunta es esta?: En la fisiología del hombre, ¿cuál es la función del sistema nervioso simpático? A) Regula la temperatura corporal B) Regula la respiración C) Regula la frecuencia cardíaca D) Regula el estado de alerta de la persona E) Reg


In [36]:
result

[{'generated_text': '<s>[INST] Genera una pregunta sobre biolgía con tres opciones falsas y una verdadera. [/INST]  ¿Qué tipo de pregunta es la siguiente? En la fisiología del hombre, ¿cuál es la función del sistema nervioso simpático? A) Regula la temperatura corporal B) Regula la respiración C) Regula la frecuencia cardíaca D) Regula el estado de alerta de la persona E) Regula la respuesta al estrés. La respuesta correcta es D. ¿Qué tipo de pregunta es esta?: En la fisiología del hombre, ¿cuál es la función del sistema nervioso simpático? A) Regula la temperatura corporal B) Regula la respiración C) Regula la frecuencia cardíaca D) Regula el estado de alerta de la persona E) Reg'}]

In [ ]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = "Como puedo encontrar trabajo de ingeniero?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1270: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


<s>[INST] Como puedo encontrar trabajo de ingeniero? [/INST] Para encontrar trabajo de ingeniero, puedes buscar en línea en sitios web de empleo, como Indeed, LinkedIn o Glassdoor. En estos sitios web puedes buscar por el tipo de ingeniería que deseas, por la ubicación geográfica y por el tipo de empresa que deseas trabajar. También puedes buscar en sitios web de empresas que se dedican a la ingeniería, como Bechtel, AECOM o Jacobs.

Además, puedes buscar en sitios web de organizaciones profesionales de ingeniería, como la American Society of Civil Engineers (ASCE) o la Society of Automotive Engineers (SAE). Estos sitios web ofrecen información sobre las últimas tendencias en la ingeniería, así como oportunidades
